# Exercise 2 · Define a safe function

You will fix a model's function list and check every call it asks for.

**Goal:** Offer only a safe function, describe it well, and check each call before it runs.

**Why it matters:** Functions let a model set off real work, such as a refund.

**From the slides:** Slide 13, "Your code runs the function; the model only asks"; Slide 14, "So the function list is a boundary"; Slide 16, "Describe it for a colleague; then watch for invented arguments".

**Before you start:**
- In function calling, the model replies with a call: a function name plus arguments (the values to pass in).
- A JSON Schema is a rule for the shape of data, such as which keys it must have.

**What the check measures:** `FUNCTIONS` offers only `get_order_status`; its description has 8 words or more, and the `order_id` description gives the format (such as `A1042` or "four digits"); `run_call` runs a right call and refuses a function not on the list (sending no refund) and a missing, extra or badly formed argument.

**Time:** about 30 minutes.

**Done when:** the last cell prints `Done:`.

## The situation

A small online shop's chat helper sends each customer message to an AI model, which replies in text or asks for a function. A teammate offered it two functions: `get_order_status`, a lookup that only reads, and `issue_refund`, an action that sends money. All data here is simulated; no model is called.

## Step 1 · Read the code that does the work

This is the shop's own code. `CODE` maps each name to its function. Run the cell.

Expect `A1042: shipped`.

In [ ]:
# Step 1: define the shop's lookup and action (simulated)
ORDERS = {"A1042": "shipped", "A1043": "delivered"}
REFUNDS_SENT = []


def get_order_status(order_id: str) -> str:
    """Return one order's status; read ORDERS, change nothing."""
    return f"{order_id}: {ORDERS.get(order_id, 'no such order')}"


def issue_refund(order_id: str, amount: float) -> str:
    """Send money back; here, only record it in REFUNDS_SENT."""
    REFUNDS_SENT.append((order_id, amount))
    return f"Refund of {amount} sent for {order_id}"


CODE = {"get_order_status": get_order_status, "issue_refund": issue_refund}
print(get_order_status("A1042"))

## Step 2 · Offer only the lookup, and describe it

`FUNCTIONS` is the function list, the only part of your code the model sees, so write it for a new colleague. The schema under `parameters` is done: `order_id` is required, no other argument is allowed, and `pattern` allows only the letter A plus four digits. In this cell:

1. Delete the whole `issue_refund` entry: it sends money, so a person decides it.
2. Replace the first `"TODO"`: what `get_order_status` returns and when to call it, in 8 words or more.
3. Replace the second `"TODO"`: the format of `order_id`, with an example.

Expect `Offered: get_order_status`.

<details><summary>Hint</summary>

The colleague cannot see your code. For `order_id`, give the letter, digit count and an example.

</details>

<details><summary>Show the answer</summary>

Delete the `issue_refund` entry and its `# TODO:` line. Then use:

```python
"description": "Returns an order's status, such as shipped. Call it only when a customer asks about an order and gives its number.",
```

```python
"description": "The order number: the letter A, then four digits, such as A1042.",
```

Why: the model sees only this list; a refund moves money, so a person decides it.

</details>

In [ ]:
# Step 2: write the function list the model sees
FUNCTIONS = [
    {
        "name": "get_order_status",
        "description": "Returns an order's status, such as shipped. Call it only when a customer asks about an order and gives its number.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "pattern": "^A[0-9]{4}$",
                    "description": "The order number: the letter A, then four digits, such as A1042.",
                },
            },
            "required": ["order_id"],
            "additionalProperties": False,
        },
    },
]
print("Offered:", ", ".join(function["name"] for function in FUNCTIONS))

## Step 3 · Check each call before your code runs it

A call is only a request. Replace the `# TODO:` lines and `return None` so that `run_call`:

1. refuses a name that is not on `FUNCTIONS`;
2. refuses arguments that break that function's schema;
3. only then runs the function from `CODE`, and returns `"Ran: "` plus its result.

A refusal is text that starts with `"Refused: "`. The last line tries a call to `issue_refund`.

Expect one line that starts with `Refused:`.

<details><summary>Hint</summary>

Check the name, then the arguments, then run. Call `validate` inside `try:`, catch `except ValidationError as error:`, and refuse with `error.message`.

</details>

<details><summary>Show the answer</summary>

Replace the `# TODO:` lines and `return None` with:

```python
    if name not in offered:
        return f"Refused: {name} is not on the function list."
    try:
        validate(arguments, offered[name]["parameters"])
    except ValidationError as error:
        return f"Refused: {error.message}"
    return "Ran: " + CODE[name](**arguments)
```

Why: your code runs the function, so it runs only a listed function whose arguments fit.

</details>

In [ ]:
# Step 3: run a call only if it passes your checks
from jsonschema import ValidationError, validate


def run_call(call: dict) -> str:
    """Run a call only if it is offered and its arguments fit; else refuse it."""
    offered = {function["name"]: function for function in FUNCTIONS}
    name, arguments = call["name"], call["arguments"]
    if name not in offered:
        return f"Refused: {name} is not on the function list."
    try:
        validate(arguments, offered[name]["parameters"])
    except ValidationError as error:
        return f"Refused: {error.message}"
    return "Ran: " + CODE[name](**arguments)


print(run_call({"name": "issue_refund", "arguments": {"order_id": "A1042", "amount": 25}}))

## Step 4 · Test a call that is right and a message that needs none

Each sample pairs a customer message with a simulated reply. A call is right for the first; your code must refuse the next two; the last needs no call, as it gives no order number. Run the cell.

Expect each message, then its result: `Ran: A1042: shipped`, then two `Refused:` lines, then `No call: Which order number?`, then `Refunds sent: []`.

In [ ]:
# Step 4: test run_call on four sample replies (simulated: written by hand)
SAMPLE_REPLIES = [
    ("Where is A1042?", {"name": "get_order_status", "arguments": {"order_id": "A1042"}}),
    ("Is A1043 late? If so, refund me 25.",
     {"name": "get_order_status", "arguments": {"order_id": "A1043", "refund_amount": 25}}),
    ("A1044 came broken. Refund my 25.", {"name": "issue_refund", "arguments": {"order_id": "A1044", "amount": 25}}),
    ("Where is my order?", "Which order number?"),
]
for message, reply in SAMPLE_REPLIES:
    result = "No call: " + reply if isinstance(reply, str) else run_call(reply)
    print(f"{message}\n    -> {result}")
print("Refunds sent:", REFUNDS_SENT)

## Check your work

Run the cell. After a fix, rerun the cells from that step down.

Expect a line that starts with `Done:`; a `Not yet:` line names the first thing to fix.

In [ ]:
# Check your work: test your list, then run_call on five new calls
import re

NEW_CALLS = [
    ("get_order_status", {"order_id": "A1043"}, "a right call", "Ran: A1043: delivered"),
    ("issue_refund", {"order_id": "A1043", "amount": 10}, "a function not on the list", "Refused:"),
    ("get_order_status", {}, "no order_id", "Refused:"),
    ("get_order_status", {"order_id": "A1043", "note": "x"}, "an extra argument", "Refused:"),
    ("get_order_status", {"order_id": "1043"}, "an order_id with no letter A", "Refused:"),
]


def first_problem() -> str:
    """Return the first problem, or "" if none."""
    names = [function["name"] for function in FUNCTIONS]
    if names != ["get_order_status"]:
        return (f"the list offers {names}; it must offer get_order_status and nothing else. "
                "In Step 2, delete only issue_refund.")
    about = str(FUNCTIONS[0]["description"])
    fmt = str(FUNCTIONS[0]["parameters"]["properties"]["order_id"]["description"])
    if len(about.split()) < 8 or "TODO" in about:
        return "in Step 2, describe get_order_status in 8 words or more: what it returns, when to call it."
    if not re.search(r"A\d{4}|\b(four|4)[ -]digit", fmt, re.IGNORECASE) or "TODO" in fmt:
        return "in Step 2, give the order_id format, with an example such as A1042."
    for name, arguments, what, start in NEW_CALLS:
        sent = len(REFUNDS_SENT)
        try:
            got = str(run_call({"name": name, "arguments": arguments}))
        except Exception as error:
            got = f"{type(error).__name__}: {getattr(error, 'message', error)}"
        if len(REFUNDS_SENT) > sent:
            return f"run_call sent a refund for {name}({arguments}). In Step 3, check first, run last."
        if not got.startswith(start):
            return f"for {name}({arguments}), {what}, run_call gave {got!r}, not {start!r}. Fix Step 3."
    return ""


try:
    problem = first_problem()
except Exception as error:
    problem = f"the check stopped on {type(error).__name__}: {error}. Run the cells above in order."
print(f"Not yet: {problem}" if problem else "Done: only the lookup is offered, and run_call runs a right call "
      "and refuses four bad ones.")

## What you learned

- Your code runs the function; the model can only ask, and only for what your list offers. So the list is a boundary: offer a lookup, and leave moving money to a person (Slides 13 to 15).
- Describe each function for a colleague (Slide 16), then check each call against its schema before your code runs it (Slide 13).
- A call is right only if its name matches exactly and every argument is allowed; test messages that need a call and ones that need none (Slide 18).

**Reflect:**

- For "Where is my order?", with no number, the model calls `get_order_status` with a guessed `A1042`. Every check passes. What could catch that?
- If staff later approve refunds, who should fill in `order_id` for `issue_refund`: the model or your code? Why? (Slide 14)